In [ ]:
import MODE_utilities as AS

import os
import pickle
import random
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# Experiment settings
# ============================================================

SUMMARY_PATH = "Raspberry_CPU_summary.pkl"

OUTPUT_DIR = "MODE_permutation_experiment_results"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True,
)

# Number of RANDOM permutations.
# Original ordering is additionally stored as permutation 0.
N_PERMUTATIONS = 1000

SEED = 89

# Same ASF weights used by MODE
ASF_WEIGHTS = np.array(
    [0.6, 0.2, 0.2],
    dtype=float,
)

# Same settings used by the Raspberry Pi CPU experiment
NORMALIZE = True
D_STAR_TP = 1.8


# ============================================================
# Reproducibility
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print("Device:", AS.device)
print("Number of random permutations:", N_PERMUTATIONS)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================
# Load saved MODE result
# ============================================================

with open(
    SUMMARY_PATH,
    "rb",
) as f:

    summary = pickle.load(f)


# ============================================================
# Inspect saved contents
# ============================================================

print(
    "Summary keys:",
    summary.keys()
)


# ============================================================
# Check required information
# ============================================================

required_keys = {
    "configuration",
    "pareto_X",
    "pareto_F",
    "history",
}

missing = (
    required_keys
    .difference(
        summary.keys()
    )
)

if missing:

    raise KeyError(
        f"Missing required keys in "
        f"{SUMMARY_PATH}: "
        f"{sorted(missing)}"
    )


# ============================================================
# Recover the MODE Problem object
#
# The summary does NOT store "problem" directly.
# Each saved NSGA-II history entry contains its Problem object.
# We use the LAST generation.
# ============================================================

history = summary[
    "history"
]

if len(history) == 0:

    raise ValueError(
        "The saved MODE history is empty."
    )


last_algorithm = history[-1]

saved_problem = (
    last_algorithm.problem
)


# ============================================================
# Make sure the saved Problem contains the trained models
# ============================================================

if not hasattr(
    saved_problem,
    "trained_models",
):

    raise AttributeError(
        "The recovered Problem object "
        "does not contain trained_models."
    )


if len(
    saved_problem.trained_models
) == 0:

    raise ValueError(
        "The recovered Problem contains "
        "no trained models."
    )


print(
    "Recovered Problem:",
    type(saved_problem)
)

print(
    "Stored trained models:",
    len(
        saved_problem.trained_models
    )
)


# ============================================================
# Reconstruct a lightweight pymoo-like result object
# from the saved Pareto front
# ============================================================

results_proxy = SimpleNamespace(

    X=np.asarray(
        summary["pareto_X"],
        dtype=float,
    ),

    F=np.asarray(
        summary["pareto_F"],
        dtype=float,
    ),
)


print(
    "Pareto X shape:",
    results_proxy.X.shape,
)

print(
    "Pareto F shape:",
    results_proxy.F.shape,
)


# ============================================================
# Apply EXACTLY the same ASF selection used by MODE
# ============================================================

(
    best_instance,
    best_model,
    best_chromosome,
    best_info,

) = AS.ApplyingASF(

    results_of_tuning=(
        results_proxy
    ),

    Problem_class=(
        saved_problem
    ),

    weights=(
        ASF_WEIGHTS
    ),

    return_full=True,
)


# ============================================================
# Move selected model to execution device
# ============================================================

best_instance.model = (
    best_instance.model.to(
        AS.device
    )
)


# ============================================================
# Recover the original offline Raspberry Pi CPU data
# directly from the saved MODE Problem
# ============================================================

offline_data = np.asarray(
    saved_problem.offline_data,
    dtype=np.float64,
)


# ============================================================
# Display recovered information
# ============================================================

print(
    "\nSaved configuration:"
)

print(
    summary["configuration"]
)


print(
    "\nRecovered ASF configuration:"
)

print(
    best_chromosome
)


print(
    "\nOffline data shape:",
    offline_data.shape,
)


# ============================================================
# Sanity checks
# ============================================================

assert offline_data.ndim == 2

assert offline_data.shape[0] == 60

assert offline_data.shape[1] == 400


for key in (
    "window",
    "batch_size",
    "hidden_dim",
    "epochs",
    "layer_dim",
    "num_clusters",
):

    assert (
        int(
            best_chromosome[key]
        )
        ==
        int(
            summary[
                "configuration"
            ][key]
        )
    )


assert np.isclose(

    float(
        best_chromosome[
            "learning_rate"
        ]
    ),

    float(
        summary[
            "configuration"
        ][
            "learning_rate"
        ]
    ),
)


print(
    "\nExact saved selected model "
    "recovered successfully."
)


In [ ]:
# ============================================================
# Normal validation performance
# ============================================================

AS._sync_device()

validation_start = (
    time.perf_counter()
)


val_output = (
    best_instance
    .validation_phase()
)


AS._sync_device()

validation_duration = (
    time.perf_counter()
    - validation_start
)


validation_rmse = float(
    val_output[0]
)

validation_r2 = (
    float(
        val_output[1]
    )
    if len(val_output) > 1
    else np.nan
)


print(
    "Validation RMSE:",
    validation_rmse,
)

print(
    "Validation R2:",
    validation_r2,
)

print(
    "Validation duration [s]:",
    validation_duration,
)

In [ ]:
# ============================================================
# Collect residual blocks ONCE
# ============================================================

residual_blocks = []

per_series_rows = []


AS._sync_device()

residual_start = (
    time.perf_counter()
)


for series_id, ts in enumerate(
    offline_data
):

    res = (
        AS.evaluate_one_individual_series(

            ML=best_instance,

            series_raw=ts,

            sdtw_normalize=(
                NORMALIZE
            ),

            return_raw_scale=False,
        )
    )

    # --------------------------------------------------------
    # Same residual definition as MODE
    # prediction - target
    # --------------------------------------------------------

    errors = (

        np.asarray(
            res["preds_used"],
            dtype=np.float64,
        )

        -

        np.asarray(
            res["targets_used"],
            dtype=np.float64,
        )
    )


    # Keep EACH time-series residual sequence separate
    residual_blocks.append(
        errors
    )


    per_series_rows.append({

        "series_id":
            int(series_id),

        "n_residuals":
            int(
                len(errors)
            ),

        "rmse_normalized":
            float(
                np.sqrt(
                    np.mean(
                        errors ** 2
                    )
                )
            ),

        "mean_residual":
            float(
                np.mean(
                    errors
                )
            ),

        "variance_residual":
            float(
                np.var(
                    errors,
                    ddof=1,
                )
            ),
    })


AS._sync_device()

residual_collection_duration = (

    time.perf_counter()
    - residual_start
)

per_series_df = (
    pd.DataFrame(
        per_series_rows
    )
)


block_lengths = np.array(

    [
        len(x)
        for x in residual_blocks
    ],

    dtype=int,
)


print(
    "Number of residual blocks:",
    len(residual_blocks),
)

print(
    "Residual block lengths:",
    np.unique(
        block_lengths
    ),
)

print(
    "Total residuals:",
    int(
        block_lengths.sum()
    ),
)

print(
    "Residual collection duration [s]:",
    residual_collection_duration,
)


# ============================================================
# Sanity checks
# ============================================================

assert (
    len(residual_blocks)
    ==
    offline_data.shape[0]
)

assert (
    len(
        np.unique(
            block_lengths
        )
    )
    == 1
)

per_series_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "per_series_residual_statistics.csv",
    ),

    index=False,
)


with open(

    os.path.join(
        OUTPUT_DIR,
        "residual_blocks.pkl",
    ),

    "wb",

) as f:

    pickle.dump(
        residual_blocks,
        f,
    )


per_series_df.head()

In [ ]:
def stability_details(
    centered_errors,
    D_star_tp=D_STAR_TP,
):

    centered_errors = np.asarray(
        centered_errors,
        dtype=np.float64,
    ).ravel()


    # ============================================================
    # Invalid / too short
    # ============================================================

    if len(
        centered_errors
    ) < 30:

        return {

            "delta_sigma2":
                1e6,

            "var_L":
                np.nan,

            "var_S":
                np.nan,

            "n_change_points":
                0,

            "n_variance_segments":
                0,

            "cp_L":
                np.nan,

            "cp_S":
                np.nan,

            "change_points":
                [],
        }


    # ============================================================
    # STEP 1
    # Change-point detection on concatenated centered residuals
    # ============================================================

    detector = (
        AS.ChangePointDetector(
            centered_errors
        )
    )

    change_points = (
        detector
        .aL_bs_genCov(
            D_star_tp
        )
    )


    all_changepoints = (

        [0]

        +

        sorted(
            change_points
        )

        +

        [
            len(
                centered_errors
            )
        ]
    )


    # ============================================================
    # Build segments between CPs
    # ============================================================

    variance_segment_pairs = []


    for i in range(

        len(
            all_changepoints
        ) - 1
    ):

        start = (
            all_changepoints[i]
        )

        end = (
            all_changepoints[
                i + 1
            ]
        )


        segment = (
            centered_errors[
                start:end
            ]
        )


        if len(segment) <= 1:
            continue


        var = np.var(
            segment,
            ddof=1,
        )


        variance_segment_pairs.append(
            (
                var,
                segment,
            )
        )


    if len(
        variance_segment_pairs
    ) == 0:

        return {

            "delta_sigma2":
                0.0,

            "var_L":
                np.nan,

            "var_S":
                np.nan,

            "n_change_points":
                0,

            "n_variance_segments":
                0,

            "cp_L":
                np.nan,

            "cp_S":
                np.nan,

            "change_points":
                [],
        }


    # ============================================================
    # STEP 2
    # Sort detected segments by variance
    # ============================================================

    variance_segment_pairs.sort(
        key=lambda x: x[0]
    )


    sorted_segments = [

        segment

        for _, segment
        in variance_segment_pairs
    ]


    # ============================================================
    # Grow from low variance / high variance side
    # ============================================================

    def grow_until_cp(
        segments,
        reverse=False,
    ):

        accumulated = []


        iterable = (

            reversed(
                segments
            )

            if reverse

            else segments
        )


        for segment in iterable:

            accumulated.append(
                segment
            )


            combined = np.concatenate(
                accumulated
            )


            detector_inner = (
                AS.ChangePointDetector(
                    combined
                )
            )


            cps = np.asarray(

                detector_inner
                .aL_bs_genCov(
                    D_star_tp
                )
            )


            valid_cps = cps[

                (cps > 0)

                &

                (
                    cps
                    <
                    len(combined) - 1
                )
            ]


            if len(
                valid_cps
            ) > 0:

                return (

                    combined,

                    int(
                        np.min(
                            valid_cps
                        )
                    ),
                )


        return (

            np.concatenate(
                accumulated
            ),

            None,
        )


    # ============================================================
    # Low variance regime L
    # ============================================================

    L, cp_L = grow_until_cp(

        sorted_segments,

        reverse=False,
    )


    # ============================================================
    # High variance regime S
    # ============================================================

    S, cp_S = grow_until_cp(

        sorted_segments,

        reverse=True,
    )


    # ============================================================
    # If no CP is eventually found
    # ============================================================

    if (
        cp_L is None
        or
        cp_S is None
    ):

        return {

            "delta_sigma2":
                0.0,

            "var_L":
                float(
                    np.var(
                        L,
                        ddof=1,
                    )
                )
                if len(L) > 1
                else np.nan,

            "var_S":
                float(
                    np.var(
                        S,
                        ddof=1,
                    )
                )
                if len(S) > 1
                else np.nan,

            "n_change_points":
                int(
                    len(
                        [
                            cp
                            for cp
                            in change_points
                            if cp > 0
                        ]
                    )
                ),

            "n_variance_segments":
                int(
                    len(
                        sorted_segments
                    )
                ),

            "cp_L":
                (
                    np.nan
                    if cp_L is None
                    else cp_L
                ),

            "cp_S":
                (
                    np.nan
                    if cp_S is None
                    else cp_S
                ),

            "change_points":
                list(
                    change_points
                ),
        }


    # ============================================================
    # Variances
    # ============================================================

    var_L = float(
        np.var(
            L,
            ddof=1,
        )
    )

    var_S = float(
        np.var(
            S,
            ddof=1,
        )
    )


    # ============================================================
    # Delta sigma squared
    # ============================================================

    delta_sigma2 = float(

        max(
            0.0,
            var_S - var_L,
        )
    )


    return {

        "delta_sigma2":
            delta_sigma2,

        "var_L":
            var_L,

        "var_S":
            var_S,

        "n_change_points":
            int(
                len(
                    [
                        cp
                        for cp
                        in change_points
                        if cp > 0
                    ]
                )
            ),

        "n_variance_segments":
            int(
                len(
                    sorted_segments
                )
            ),

        "cp_L":
            int(cp_L),

        "cp_S":
            int(cp_S),

        "change_points":
            list(
                change_points
            ),
    }

In [ ]:
# ============================================================
# Original ordering
# ============================================================

n_series = len(
    residual_blocks
)


original_order = np.arange(
    n_series,
    dtype=int,
)


# ============================================================
# Concatenate residuals in ORIGINAL order
# ============================================================

original_errors = np.concatenate(

    [
        residual_blocks[i]
        for i
        in original_order
    ]
)


# ============================================================
# Center AFTER concatenation
# Same as MODE
# ============================================================

original_centered = (

    original_errors
    -
    np.mean(
        original_errors
    )
)


# ============================================================
# RMSE of pooled residuals
# ============================================================

original_pooled_rmse = float(

    np.sqrt(

        np.mean(
            original_errors ** 2
        )
    )
)


original_centered_rms = float(

    np.sqrt(

        np.mean(
            original_centered ** 2
        )
    )
)

# ============================================================
# Stability score
# ============================================================

original_details = (
    stability_details(
        original_centered
    )
)


# ============================================================
# Compare against MODE's ORIGINAL implementation
# ============================================================

official_original_delta = float(

    saved_problem
    ._variance_instability_score(
        original_centered
    )
)


assert np.isclose(

    original_details[
        "delta_sigma2"
    ],

    official_original_delta,

    rtol=1e-12,
    atol=1e-12,
)

In [ ]:
# These will be used to prove that permutations
# contain EXACTLY the same residual values.

BASE_SUM = float(
    np.sum(
        original_errors
    )
)

BASE_SUMSQ = float(
    np.sum(
        original_errors ** 2
    )
)


print(
    "Original pooled residual RMSE:",
    original_pooled_rmse,
)

print(
    "Original centered residual RMS:",
    original_centered_rms,
)

print(
    "Original Delta sigma^2:",
    original_details[
        "delta_sigma2"
    ],
)

print(
    "Original detected change points:",
    original_details[
        "n_change_points"
    ],
)

print(
    "\nExact MODE stability implementation check: PASSED"
)

In [ ]:
# ============================================================
# Random generator
# ============================================================

rng = np.random.default_rng(
    SEED
)


rows = []


# ============================================================
# Permutation 0 = ORIGINAL ORDER
# ============================================================

rows.append({

    "permutation_id":
        0,

    "is_original":
        True,

    "order":
        ",".join(
            map(
                str,
                original_order
            )
        ),

    "delta_sigma2":
        float(
            original_details[
                "delta_sigma2"
            ]
        ),

    "pooled_residual_rmse":
        original_pooled_rmse,

    "centered_residual_rms":
        original_centered_rms,

    "n_change_points":
        int(
            original_details[
                "n_change_points"
            ]
        ),

    "n_variance_segments":
        int(
            original_details[
                "n_variance_segments"
            ]
        ),

    "var_L":
        float(
            original_details[
                "var_L"
            ]
        ),

    "var_S":
        float(
            original_details[
                "var_S"
            ]
        ),

    "cp_L":
        original_details[
            "cp_L"
        ],

    "cp_S":
        original_details[
            "cp_S"
        ],
})


seen_orders = {

    tuple(
        original_order.tolist()
    )
}

# ============================================================
# Start timer
# ============================================================

permutation_start = (
    time.perf_counter()
)


# ============================================================
# Random permutations
# ============================================================

for permutation_id in range(

    1,
    N_PERMUTATIONS + 1,
):


    # --------------------------------------------------------
    # Generate a NEW unique ordering
    # --------------------------------------------------------

    while True:

        order = (
            rng.permutation(
                n_series
            )
        )


        order_key = tuple(
            order.tolist()
        )


        if order_key not in (
            seen_orders
        ):

            seen_orders.add(
                order_key
            )

            break


    # --------------------------------------------------------
    # Concatenate SAME residual blocks
    # in a DIFFERENT order
    # --------------------------------------------------------

    errors = np.concatenate(

        [
            residual_blocks[i]

            for i
            in order
        ]
    )


    # ========================================================
    # VERY IMPORTANT:
    # prove that residual values have not changed
    # ========================================================

    if not np.isclose(

        np.sum(errors),

        BASE_SUM,

        rtol=1e-12,
        atol=1e-12,
    ):

        raise RuntimeError(
            "Residual sum changed after permutation."
        )


    if not np.isclose(

        np.sum(
            errors ** 2
        ),

        BASE_SUMSQ,

        rtol=1e-12,
        atol=1e-12,
    ):

        raise RuntimeError(
            "Residual sum of squares changed "
            "after permutation."
        )


    # --------------------------------------------------------
    # Center AFTER concatenating
    # --------------------------------------------------------

    centered_errors = (

        errors
        -
        np.mean(errors)
    )


    # --------------------------------------------------------
    # Stability calculation
    # --------------------------------------------------------

    details = (
        stability_details(
            centered_errors
        )
    )


    # --------------------------------------------------------
    # RMSE
    # --------------------------------------------------------

    pooled_rmse = float(

        np.sqrt(

            np.mean(
                errors ** 2
            )
        )
    )


    centered_rms = float(

        np.sqrt(

            np.mean(
                centered_errors ** 2
            )
        )
    )


    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    rows.append({

        "permutation_id":
            int(
                permutation_id
            ),

        "is_original":
            False,

        "order":
            ",".join(
                map(
                    str,
                    order
                )
            ),

        "delta_sigma2":
            float(
                details[
                    "delta_sigma2"
                ]
            ),

        "pooled_residual_rmse":
            pooled_rmse,

        "centered_residual_rms":
            centered_rms,

        "n_change_points":
            int(
                details[
                    "n_change_points"
                ]
            ),

        "n_variance_segments":
            int(
                details[
                    "n_variance_segments"
                ]
            ),

        "var_L":
            float(
                details[
                    "var_L"
                ]
            ),

        "var_S":
            float(
                details[
                    "var_S"
                ]
            ),

        "cp_L":
            details[
                "cp_L"
            ],

        "cp_S":
            details[
                "cp_S"
            ],
    })


    # ========================================================
    # Save checkpoint every 25 permutations
    # ========================================================

    if (

        permutation_id % 25 == 0

        or

        permutation_id
        ==
        N_PERMUTATIONS

    ):

        elapsed = (

            time.perf_counter()
            -
            permutation_start
        )


        print(

            f"{permutation_id}/"
            f"{N_PERMUTATIONS} permutations "
            f"completed | "
            f"elapsed = {elapsed:.2f} s"
        )


        pd.DataFrame(
            rows
        ).to_csv(

            os.path.join(
                OUTPUT_DIR,
                "permutation_results.csv",
            ),

            index=False,
        )


# ============================================================
# Total permutation duration
# ============================================================

permutation_duration = (

    time.perf_counter()
    -
    permutation_start
)


permutation_df = (
    pd.DataFrame(
        rows
    )
)


print(
    "\nPermutation duration [s]:",
    permutation_duration,
)


permutation_df.head()

In [ ]:
# ============================================================
# Random permutations only
# ============================================================

random_df = (

    permutation_df[
        ~permutation_df[
            "is_original"
        ]
    ]
    .copy()
)


random_delta = (

    random_df[
        "delta_sigma2"
    ]

    .to_numpy(
        dtype=float
    )
)


original_delta = float(

    original_details[
        "delta_sigma2"
    ]
)

# ============================================================
# Where does original ordering lie
# relative to random orderings?
# ============================================================

original_percentile = float(

    100.0

    *

    np.mean(

        random_delta
        <=
        original_delta
    )
)


delta_mean = float(
    np.mean(
        random_delta
    )
)

delta_std = float(
    np.std(
        random_delta,
        ddof=1,
    )
)

delta_median = float(
    np.median(
        random_delta
    )
)

delta_min = float(
    np.min(
        random_delta
    )
)

delta_max = float(
    np.max(
        random_delta
    )
)

delta_p05 = float(
    np.quantile(
        random_delta,
        0.05,
    )
)

delta_p95 = float(
    np.quantile(
        random_delta,
        0.95,
    )
)

rmse_min = float(

    permutation_df[
        "pooled_residual_rmse"
    ].min()
)


rmse_max = float(

    permutation_df[
        "pooled_residual_rmse"
    ].max()
)


rmse_range = float(

    rmse_max
    -
    rmse_min
)


summary_stats = {

    "n_series":
        int(
            n_series
        ),

    "residuals_per_series":
        int(
            block_lengths[0]
        ),

    "total_residuals":
        int(
            block_lengths.sum()
        ),

    "n_random_permutations":
        int(
            N_PERMUTATIONS
        ),

    "validation_rmse":
        float(
            validation_rmse
        ),

    "original_pooled_residual_rmse":
        float(
            original_pooled_rmse
        ),

    "original_delta_sigma2":
        original_delta,

    "random_delta_mean":
        delta_mean,

    "random_delta_std":
        delta_std,

    "random_delta_median":
        delta_median,

    "random_delta_min":
        delta_min,

    "random_delta_max":
        delta_max,

    "random_delta_p05":
        delta_p05,

    "random_delta_p95":
        delta_p95,

    "original_delta_percentile":
        original_percentile,

    "pooled_rmse_min":
        rmse_min,

    "pooled_rmse_max":
        rmse_max,

    "pooled_rmse_range":
        rmse_range,

    "permutation_duration_sec":
        float(
            permutation_duration
        ),
}

summary_df = pd.DataFrame(
    [
        summary_stats
    ]
)


summary_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "permutation_summary.csv",
    ),

    index=False,
)


summary_df.T

In [ ]:
summary_txt = os.path.join(

    OUTPUT_DIR,

    "permutation_summary.txt",
)


with open(

    summary_txt,

    "w",

    encoding="utf-8",

) as f:


    f.write(
        "MODE RESIDUAL-CONCATENATION "
        "PERMUTATION EXPERIMENT\n"
    )

    f.write(
        "=" * 72
        + "\n"
    )


    # ========================================================
    # MODE configuration
    # ========================================================

    f.write(
        "\nFIXED MODE CONFIGURATION\n"
    )

    f.write(
        "-" * 72
        + "\n"
    )


    for key, value in (
        best_chromosome.items()
    ):

        f.write(
            f"{key}: {value}\n"
        )


    f.write(
        f"ASF weights: "
        f"{ASF_WEIGHTS.tolist()}\n"
    )

    f.write(
        f"D_star_tp: "
        f"{D_STAR_TP}\n"
    )


    # ========================================================
    # Residual information
    # ========================================================

    f.write(
        "\nDATA / RESIDUALS\n"
    )

    f.write(
        "-" * 72
        + "\n"
    )


    f.write(
        f"Number of time series: "
        f"{n_series}\n"
    )

    f.write(
        f"Offline series length: "
        f"{offline_data.shape[1]}\n"
    )

    f.write(
        f"Residuals per series: "
        f"{block_lengths[0]}\n"
    )

    f.write(
        f"Total residuals: "
        f"{block_lengths.sum()}\n"
    )


    # ========================================================
    # Accuracy
    # ========================================================

    f.write(
        "\nFIXED MODEL ACCURACY\n"
    )

    f.write(
        "-" * 72
        + "\n"
    )


    f.write(
        f"Validation RMSE: "
        f"{validation_rmse:.12f}\n"
    )

    f.write(
        f"Original pooled residual RMSE: "
        f"{original_pooled_rmse:.12f}\n"
    )

    f.write(
        f"Pooled RMSE range across all orders: "
        f"{rmse_range:.16e}\n"
    )


    # ========================================================
    # Stability
    # ========================================================

    f.write(
        "\nDELTA SIGMA^2 ORDER SENSITIVITY\n"
    )

    f.write(
        "-" * 72
        + "\n"
    )


    f.write(
        f"Random permutations: "
        f"{N_PERMUTATIONS}\n"
    )

    f.write(
        f"Original Delta sigma^2: "
        f"{original_delta:.12f}\n"
    )

    f.write(
        f"Random mean: "
        f"{delta_mean:.12f}\n"
    )

    f.write(
        f"Random std: "
        f"{delta_std:.12f}\n"
    )

    f.write(
        f"Random median: "
        f"{delta_median:.12f}\n"
    )

    f.write(
        f"Random min: "
        f"{delta_min:.12f}\n"
    )

    f.write(
        f"Random max: "
        f"{delta_max:.12f}\n"
    )

    f.write(
        f"Random 5th percentile: "
        f"{delta_p05:.12f}\n"
    )

    f.write(
        f"Random 95th percentile: "
        f"{delta_p95:.12f}\n"
    )

    f.write(
        f"Original-order percentile "
        f"among random permutations: "
        f"{original_percentile:.4f}%\n"
    )


    # ========================================================
    # Timing
    # ========================================================

    f.write(
        "\nTIMING\n"
    )

    f.write(
        "-" * 72
        + "\n"
    )


    f.write(
        f"Validation duration [s]: "
        f"{validation_duration:.10f}\n"
    )

    f.write(
        f"Residual collection duration [s]: "
        f"{residual_collection_duration:.10f}\n"
    )

    f.write(
        f"Permutation-analysis duration [s]: "
        f"{permutation_duration:.10f}\n"
    )


print(
    "Saved:",
    summary_txt,
)

In [ ]:
plt.figure(
    figsize=(10, 5)
)


plt.scatter(

    random_df[
        "permutation_id"
    ],

    random_df[
        "delta_sigma2"
    ],

    s=16,

    alpha=0.6,

    label=(
        "Random permutations"
    ),
)


plt.axhline(

    original_delta,

    linestyle="--",

    linewidth=2,

    label=(
        "Original order"
    ),
)


plt.xlabel(
    "Permutation ID"
)

plt.ylabel(
    r"$\Delta\sigma^2$"
)

plt.legend()

plt.tight_layout()


plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "delta_sigma2_by_permutation.pdf",
    ),

    bbox_inches="tight",
)


plt.show()

In [ ]:
plt.figure(
    figsize=(8, 5)
)


plt.hist(

    random_delta,

    bins=30,

    alpha=0.75,
)


plt.axvline(

    original_delta,

    linestyle="--",

    linewidth=2,

    label=(
        "Original order"
    ),
)


plt.xlabel(
    r"$\Delta\sigma^2$"
)

plt.ylabel(
    "Frequency"
)

plt.legend()

plt.tight_layout()


plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "delta_sigma2_distribution.pdf",
    ),

    bbox_inches="tight",
)


plt.show()

In [ ]:
plt.figure(
    figsize=(10, 4)
)


plt.plot(

    permutation_df[
        "permutation_id"
    ],

    permutation_df[
        "pooled_residual_rmse"
    ],
)


plt.xlabel(
    "Permutation ID"
)

plt.ylabel(
    "Pooled residual RMSE"
)

plt.tight_layout()


plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "pooled_rmse_by_permutation.pdf",
    ),

    bbox_inches="tight",
)


plt.show()

In [ ]:
print(
    "=" * 72
)

print(
    "FINAL PERMUTATION EXPERIMENT RESULTS"
)

print(
    "=" * 72
)


print(
    "\nValidation RMSE:",
    validation_rmse,
)


print(
    "\nOriginal Delta sigma^2:",
    original_delta,
)


print(
    "\nRandom Delta sigma^2 "
    "mean ± std:",
    delta_mean,
    "±",
    delta_std,
)


print(
    "\nRandom Delta sigma^2 range:",
    delta_min,
    "to",
    delta_max,
)


print(
    "\nOriginal-order percentile:",
    original_percentile,
)


print(
    "\nPooled RMSE range "
    "across all orders:",
    rmse_range,
)

if rmse_range < 1e-12:

    print(
        "\nRMSE invariance check: PASSED."
    )

else:

    print(
        "\nRMSE invariance check: "
        "inspect numerical differences."
    )


print(
    "\nResults saved in:",
    OUTPUT_DIR,
)

# Statistical test on permutation

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist

# Load already-computed permutation results
df = pd.read_csv("permutation_results.csv")

# Keep only the 1000 random permutations
df = df[df["is_original"] == False].reset_index(drop=True)

# Convert saved order strings to arrays
orders = np.array([
    [int(x) for x in order.split(",")]
    for order in df["order"]
])

delta = df["delta_sigma2"].to_numpy()

# Convert each permutation to the position of every series
positions = np.empty_like(orders)

for i, order in enumerate(orders):
    positions[i, order] = np.arange(len(order))

# Pairwise distance between orderings
order_dist = pdist(
    positions.astype(float),
    metric="euclidean"
)

# Pairwise difference between Delta sigma^2 values
delta_dist = pdist(
    delta[:, None],
    metric="cityblock"
)

# Observed Mantel correlation
r_obs = np.corrcoef(
    order_dist,
    delta_dist
)[0, 1]

# Permutation significance test
rng = np.random.default_rng(89)

N_PERM = 999
count = 0

for _ in range(N_PERM):

    delta_perm = rng.permutation(delta)

    delta_dist_perm = pdist(
        delta_perm[:, None],
        metric="cityblock"
    )

    r_perm = np.corrcoef(
        order_dist,
        delta_dist_perm
    )[0, 1]

    if abs(r_perm) >= abs(r_obs):
        count += 1

p_value = (count + 1) / (N_PERM + 1)

print(f"Mantel correlation: r = {r_obs:.6f}")
print(f"Permutation p-value: p = {p_value:.4f}")